# all-reduce-eval-metrics — worked example 1: Sync top-1 accuracy across ranks with one packed all_reduce

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `all-reduce-eval-metrics`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

After each rank evaluates its shard of the test set you have a local `(correct, seen)` pair. Accuracy is *not* the mean of per-rank accuracies when shards differ in size — you must sum the raw counts globally first, then divide once. Packing `correct` and `seen` into one length-2 tensor lets a single `all_reduce(SUM)` carry both, halving the network round-trips versus two reduces.

## Worked solution

**Step 1 — simulate per-rank eval results.** We pretend `world_size=4` ranks each saw a different shard: counts `seen=[40, 40, 40, 8]` (last shard is the ragged tail) and `correct=[30, 36, 28, 8]`. Real DDP would compute these from the model; here we hard-code them so the demo is deterministic.

**Step 2 — why naive averaging is wrong.** Per-rank accuracies are `0.75, 0.90, 0.70, 1.00`; their plain mean is `0.8375`. But the small 8-sample shard is over-weighted. The honest global accuracy is `(30+36+28+8) / (40+40+40+8) = 102/128 = 0.796875`.

**Step 3 — pack into one float tensor.** Each rank builds `stats = t.tensor([float(correct), float(seen)])`. Counts must be cast to float because `all_reduce` operates on float tensors (an int tensor would error on gloo for some ops, and float keeps the API uniform).

**Step 4 — a single SUM all_reduce.** `dist.all_reduce(stats, SUM)` replaces every rank's `stats` with the element-wise sum across ranks. Afterward `stats[0]` is total correct (102) and `stats[1]` is total seen (128) — identical on every rank.

**Step 5 — divide once, on every rank.** `acc = stats[0] / stats[1]` gives the true global accuracy. Dividing after the reduce (not before) is what makes ragged shards correct. Because we mock `all_reduce` as an in-process sum, the demo runs single-process but the arithmetic is exactly what real DDP produces.

In [ ]:
class MockDist:
    class ReduceOp:
        SUM = 'sum'
    def __init__(self, rank_tensors):
        # rank_tensors: list of the per-rank tensors that would each call all_reduce
        self.rank_tensors = rank_tensors
    def all_reduce(self, tensor, op=ReduceOp.SUM):
        total = t.zeros_like(self.rank_tensors[0])
        for rt in self.rank_tensors:
            total += rt
        tensor.copy_(total)

def global_accuracy(dist_module, local_correct, local_seen):
    stats = t.tensor([float(local_correct), float(local_seen)], dtype=t.float32)
    dist_module.all_reduce(stats, op=dist_module.ReduceOp.SUM)
    return (stats[0] / stats[1]).item()

t.manual_seed(0)
correct = [30, 36, 28, 8]
seen = [40, 40, 40, 8]
rank_tensors = [t.tensor([float(c), float(s)], dtype=t.float32) for c, s in zip(correct, seen)]
mock = MockDist(rank_tensors)
acc = global_accuracy(mock, correct[0], seen[0])
print('global accuracy:', round(acc, 6))
print('naive mean of accuracies:', round(sum(c / s for c, s in zip(correct, seen)) / len(seen), 6))